# 02 - Análise Exploratória e Qualidade de Dados
**Squad 1 — Data Quality em Tempo Real | Dupla 1**
**Integrantes:** Gabriel Franz Simoni & Carlos Eduardo Santos de Souza
**Tabelas:** `ecommerce_itens_pedido` e `ecommerce_rastreamento_entregas`

### Objetivo (Task 3):
Consolidar todas as janelas de tempo real disponíveis, remover duplicatas, e aplicar análise de qualidade (completude, duplicidade, cardinalidade, estatísticas).

**Pré-requisito:** `01_conexao_adls.ipynb` já executado nessa sessão (ou execute a célula de credenciais abaixo antes de continuar).

In [0]:
adls_client_id = dbutils.secrets.get(scope="internship-squad1", key="adls-client-id")
adls_tenant_id = dbutils.secrets.get(scope="internship-squad1", key="adls-tenant-id")
adls_client_secret = dbutils.secrets.get(scope="internship-squad1", key="adls-client-secret")
adls_storage_account = dbutils.secrets.get(scope="internship-squad1", key="adls-storage-account")

adls_options = {
    f"fs.azure.account.auth.type.{adls_storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{adls_storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{adls_storage_account}.dfs.core.windows.net": adls_client_id,
    f"fs.azure.account.oauth2.client.secret.{adls_storage_account}.dfs.core.windows.net": adls_client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{adls_storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{adls_tenant_id}/oauth2/token",
}

In [0]:
caminho_itens_pedido = f"abfss://raw@{adls_storage_account}.dfs.core.windows.net/real-time-data/*/*/*/*/ecommerce_itens_pedido.parquet"

df_itens_pedido_raw = spark.read.options(**adls_options).parquet(caminho_itens_pedido)
df_itens_pedido_rt = df_itens_pedido_raw.dropDuplicates(["id_item_pedido"])

print(f"itens_pedido: {df_itens_pedido_raw.count()} linhas brutas -> {df_itens_pedido_rt.count()} únicas")

In [0]:
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient
import pandas as pd
from io import BytesIO

credential = ClientSecretCredential(adls_tenant_id, adls_client_id, adls_client_secret)
service_client = DataLakeServiceClient(
    account_url=f"https://{adls_storage_account}.dfs.core.windows.net",
    credential=credential,
)
file_system_client = service_client.get_file_system_client(file_system="raw")

arquivos_rastreamento = [
    p.name for p in file_system_client.get_paths(path="real-time-data", recursive=True)
    if p.name.endswith("ecommerce_rastreamento.parquet")
]

dfs_rastreamento = []
for caminho in arquivos_rastreamento:
    file_client = file_system_client.get_file_client(caminho)
    conteudo = file_client.download_file().readall()
    dfs_rastreamento.append(pd.read_parquet(BytesIO(conteudo)))

pdf_rastreamento = pd.concat(dfs_rastreamento, ignore_index=True)
df_rastreamento_raw = spark.createDataFrame(pdf_rastreamento)
df_rastreamento_rt = df_rastreamento_raw.dropDuplicates(["id_rastreamento"])

print(f"rastreamento: {df_rastreamento_raw.count()} linhas brutas (via SDK, {len(arquivos_rastreamento)} janelas) -> {df_rastreamento_rt.count()} únicas")

In [0]:
from pyspark.sql.functions import col, count, when, countDistinct

def analisar_qualidade(df, nome_tabela, chave_primaria):
    print(f"\n{'='*60}")
    print(f"ANÁLISE DE QUALIDADE: {nome_tabela}")
    print(f"{'='*60}")

    total_linhas = df.count()
    print(f"\nTotal de linhas: {total_linhas}")

    print("\n--- Completude (nulos por coluna) ---")
    df.select([
        count(when(col(c).isNull(), c)).alias(c) for c in df.columns
    ]).show()

    print(f"--- Duplicidade (por chave '{chave_primaria}') ---")
    duplicatas = total_linhas - df.dropDuplicates([chave_primaria]).count()
    print(f"Registros duplicados: {duplicatas}")

    print("\n--- Cardinalidade (valores distintos por coluna) ---")
    df.select([
        countDistinct(col(c)).alias(c) for c in df.columns
    ]).show()

    print("--- Estatísticas descritivas ---")
    df.describe().show()

    return {"tabela": nome_tabela, "total_linhas": total_linhas, "duplicatas": duplicatas}


resultado_itens = analisar_qualidade(df_itens_pedido_rt, "ecommerce_itens_pedido", "id_item_pedido")
resultado_rastreamento = analisar_qualidade(df_rastreamento_rt, "ecommerce_rastreamento_entregas", "id_rastreamento")

In [0]:
# ============================================================
# ACHADOS DE QUALIDADE: registros com valores inconsistentes
# Nenhum dado é alterado aqui — só identificação e evidência,
# conforme escopo da fase atual (Card: Análise Exploratória).
# ============================================================

# 1. Preço unitário negativo (não deveria existir; desconto já
#    tem coluna própria, então preço não deveria compensar isso)
print("=== Itens com preço unitário negativo ===")
df_itens_pedido_rt.filter(df_itens_pedido_rt.preco_unitario < 0).show(truncate=False)

# 2. Quantidade igual a zero (item de pedido sem nenhuma unidade
#    não faz sentido de negócio)
print("\n=== Itens com quantidade zero ===")
df_itens_pedido_rt.filter(df_itens_pedido_rt.quantidade == 0).show(truncate=False)

# 3. Desconto aplicado maior que o próprio preço unitário
#    (resultaria em valor final negativo na prática)
print("\n=== Itens com desconto maior que o preço unitário ===")
df_itens_pedido_rt.filter(
    df_itens_pedido_rt.desconto_aplicado > df_itens_pedido_rt.preco_unitario
).show(truncate=False)